In [ ]:
# MolCalib training on Kaggle. Code and splits come from two attached datasets.
import glob
import os
import subprocess
import sys
import zipfile


def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"not found under /kaggle/input: {pattern}"
    return hits[0]

SPLITS_DIR = os.path.dirname(find("caco2_wang_seed1.csv"))
zips = glob.glob("/kaggle/input/**/molcalib_code.zip", recursive=True)
if zips:
    CODE = "/kaggle/working/code"
    zipfile.ZipFile(zips[0]).extractall(CODE)
else:  # the platform may have extracted the zip already
    CODE = os.path.dirname(find("molcalib/__init__.py"))
with open(os.path.join(CODE, "commit.txt")) as f:
    COMMIT = f.read().strip()
print("splits:", SPLITS_DIR, "| code:", CODE, "| commit:", COMMIT)

In [ ]:
# Kaggle's preinstalled CUDA torch is kept: requirements-kaggle.txt has no torch line.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                os.path.join(CODE, "requirements-kaggle.txt")], check=True)
import chemprop
import rdkit
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "gpus", torch.cuda.device_count(), "| chemprop", chemprop.__version__, "| rdkit", rdkit.__version__)
assert torch.cuda.is_available(), "no GPU: enable it in the notebook settings"

In [ ]:
EXTRA = [a for a in ["__EXTRA_ARGS__"] if a]  # set by scripts/kaggle_ctl.py ("--smoke" or empty)
env = dict(os.environ, PYTHONPATH=CODE, MOLCALIB_COMMIT=COMMIT)
os.makedirs("/kaggle/working/results", exist_ok=True)

def launch(tag, device, seeds=()):
    cmd = [sys.executable, os.path.join(CODE, "scripts", "train_all.py"),
           "--device", device, "--splits", SPLITS_DIR,
           "--out", "/kaggle/working/cache/preds",
           "--compute-csv", f"/kaggle/working/results/compute_{tag}.csv",
           "--log", f"/kaggle/working/progress_{tag}.log", *EXTRA]
    if seeds:
        cmd += ["--seeds", *map(str, seeds)]
    return subprocess.Popen(cmd, cwd="/kaggle/working", env=env)

if "--smoke" not in EXTRA and torch.cuda.device_count() >= 2:
    procs = [launch("gpu0", "cuda:0", (1, 2, 3)), launch("gpu1", "cuda:1", (4, 5))]
else:
    procs = [launch("gpu0", "cuda:0")]
print("exit codes:", [p.wait() for p in procs])